# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](./coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

------------------------------
#### E1:

No that doesn't.. 100 successful demonstrations provide strong evidence that the robot can perform the task under the conditions represented in those runs.. BUT that doesn't automatically mean it can do the same across different home environments.. where things can change and uncertainty plays a big role.. a wider competence claim needs a clearly defined range of conditions, varied tests, checked outcomes and evidence of failures as well as successes.. NOT just a big number of successful trials.

This is especially important if the demonstrations involve human teleoperation.. since a human successfully controlling the robot doesn't prove that its AI model can perform the task autonomously.

**My confidence: 5/5**



#### E2:

The nearly full mug is more likely to spill during the sharp turn because the liquid has less free space above it (as mentioned in the lec) and moves towards the outer side due to the turn's acceleration.. the **untested** assumption is that a motion successful with a half-full mug will also succeed with a nearly full mug. This is as if like treating compotence as an abstract property independent of world state.



#### E3:

I would choose **"ask"**.. because the acceptance of the use of the white mug is unconfirmed by **Eva** yet.
Also commitment check here shouldn't assume that availability automatically means acceptability...



#### E4:

According to the **Lecture** ---> Explain (Why did it happen?) -> Predict (What will happen?) -> Design (How can we improve it?) -> Bound (What are the limits?) -> Check (How do we verify success?)

sooo.. formulating a scenario here that would adhere the **understanding** operationally and distinctively would be like this:

1. **Explain**:- Why did the robot manage to carry Eva's half-full mug around the sharp turn without spilling? Was it because of the speed, the turning motion or the available free space in the mug?
2. **Predict**:- If the same mug is now nearly full and the robot follows the exact same route and motion.. is the coffee likely to spill during the turn?
3. **Design**:- Would using a tray or reducing the robot's speed during turns make carrying the coffee safer?
4. **Bound**:- Under which conditions.. like for ex. (mug types, fill levels, routes and robot bodies) can the robot reliably deliver coffee without spilling?
5. **Check**:- What evidence confirms that Eva received the correct coffee safely and that the task was actually completed?



#### E5:

##### Core Functional Modules mapped onto the Integration Diagram

| Functional module | Integration Diagram box        | Role in Eva's coffee scenario                                                                         |
| ----------------- | ------------------------------ | ----------------------------------------------------------------------------------------------------- |
| Interpreting      | Interpret & ground (L10, L13)  | Understand Eva's request and identify what she means by "my coffee".                                  |
| Perceiving        | Perceive (L11)                 | Observe the environment, identify available mugs, Eva's position, and possible obstacles.             |
| Proposing         | Propose (L7, L14)              | Generate possible actions, such as selecting the white mug and carrying it to Eva.                    |
| Predicting        | Know & predict (L4–5, L13)     | Predict the possible consequences of an action, such as whether the coffee might spill during a turn. |
| Warranting        | Decide: warrant gate (L6, L9)  | Check whether sufficient evidence exists to justify proceeding with the proposed action.              |
| Acting            | Act & regulate (L9, L12)       | Execute and regulate the robot's movements while adapting to changes.                                 |
| Recording         | Record & justify (L3, L9, L14) | Record the request, assumptions, decisions, observations, and outcomes for later evaluation.          |

***Excluding the Body–World System***



#### E6:

If **Eva** just suddenly stands up.. the robot shouldn't just continue following the original path because the handover position it planned for is maybe no longer suitable.. I would have the robot pause, keep the mug stable and then observe Eva's new position before deciding how to continue.. it could then find a new route and choose a more suitable handover position for **Eva**.

The main risk of continuing with the original plan is that the robot might collide with **Eva** or her walking frame because it is relying on outdated information about the environment.. alsooo making like any sudden turn or movement to adjust to Eva could cause the coffee to spill or mess up with the whole end goal trajectory planning.

Before committing to the revised path.. the robot needs fresh sensor observations confirming **Eva's** current position, the location of her walking frame and that the new route is clear.. it should also establish that the planned movements allow it to carry the mug without excessive tilting or sudden acceleration and that the new handover position is safely reachable by **Eva**.. I think if these conditions cannot be confirmed.. the robot then should wait rather than proceed.

------------------------------

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [21]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from collections import Counter
from statistics import mean, pvariance
from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"
    ## Question 2.1
    contact_depth_m: float = 0.0
    ## Question 4.1
    trajectory_distance_m: float = 10.0

## Question 1.2
class Proposer:
    """
    System 1 (The Proposer): PrameterPacket utilizer?
    generates candidate execution parameters using
    independent.. truncated "Gaussian" distributions
    """

    def __init__(
        self,
        velocity_mean: float = 0.35,
        velocity_std: float = 0.07,
        tilt_mean: float = 0.015,
        tilt_std: float = 0.004,
        min_velocity: float = 0.05,
        max_velocity: float = 0.8,
        min_tilt: float = 0.005,
        max_tilt: float = 0.05,
        grasp_pose: Tuple[float, float, float] = (0.45, 0.12, 0.30),
        seed: Optional[int] = None,
        ## Question 2.2
        contact_depth_m: float = 0.03
    ):
        self.velocity_mean = velocity_mean
        self.velocity_std = velocity_std
        self.tilt_mean = tilt_mean
        self.tilt_std = tilt_std

        self.min_velocity = min_velocity
        self.max_velocity = max_velocity
        self.min_tilt = min_tilt
        self.max_tilt = max_tilt

        self.grasp_pose = grasp_pose

        self.rng = random.Random(seed)

        ## Question 2.2
        self.contact_depth_m = contact_depth_m

        if velocity_std <= 0 or tilt_std <= 0:
            raise ValueError("Standard deviations must be positive.")

        if not (0 < min_velocity < max_velocity):
            raise ValueError("Invalid velocity limits.")

        if not (0 < min_tilt < max_tilt):
            raise ValueError("Invalid tilt limits.")

    def _sample_gaussian(self, mean: float, std: float, lower: float, upper: float) -> float:
        """sample from a Gaussian distribution within given bounds"""

        for _ in range(100):
            value = self.rng.gauss(mean, std)

            if lower <= value <= upper:
                return value

        ## if repeated sampling produces no valid value just fallback to the min
        return min(max(mean, lower), upper)

    def propose(self, goal: GoalSpecification) -> ParameterPacket:
        """generate a candidate set of execution parameters"""

        ## do NOT exceed the maximum velocity allowed by the interpreted goal
        velocity_upper = min(self.max_velocity, goal.max_nav_velocity_mps)

        if velocity_upper < self.min_velocity:
            raise ValueError("No feasible positive velocity available.")

        ## keeping the preferred velocity below the available upper limit
        velocity_mean = max(self.min_velocity, min(self.velocity_mean, 0.7 * velocity_upper))

        ## mov velocity sample candidate
        max_velocity = self._sample_gaussian(
            velocity_mean,
            self.velocity_std,
            self.min_velocity,
            velocity_upper
        )

        ## mug tilt rate sample candidate 
        tilt_rate = self._sample_gaussian(
            self.tilt_mean,
            self.tilt_std,
            self.min_tilt,
            self.max_tilt
        )

        return ParameterPacket(
            grasp_pose=self.grasp_pose,
            max_velocity_mps=max_velocity,
            tilt_rate_radps=tilt_rate,
            source_generator="truncated_gaussian",
            ## Question 2.2
            contact_depth_m=self.contact_depth_m
        )

@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"
    ## Question 4.1
    battery_level_percent: float = 80.0
    battery_capacity_wh: float = 100.0
    battery_reserve_percent: float = 5.0


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        ## Question 2.1
        stiffness_coefficient = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        
        if params.contact_depth_m > 0.02:
            excess_depth = params.contact_depth_m - 0.02
            stiffness_coefficient *= 1.8 ** (excess_depth / 0.01)
        
        predicted_force = stiffness_coefficient * (params.max_velocity_mps ** 1.5)

        ## Question 3.1
        temperature_difference = max(0.0, self.belief.ambient_temperature_c - 20.0)
        thermal_factor = max(0.01, 1.0 - temperature_difference * 0.01)
        effective_viscosity = (self.belief.fluid_viscosity_pas * thermal_factor)
        
        radius = 0.05
        wave_height_m = (params.tilt_rate_radps * radius) / effective_viscosity
        predicted_wave = wave_height_m * 1000.0

        # Injecting minor epistemic simulation error (Sim-to-Real gap)
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(max(0.0, predicted_force + epistemic_force_error), 2),
            "predicted_wave_mm": round(max(0.0, predicted_wave + epistemic_wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }


class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt
        self.energy_wh_per_meter = 1.0

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        
        estimated_energy_wh = (params.trajectory_distance_m * self.energy_wh_per_meter)
        estimated_drain_percent = (estimated_energy_wh / belief.battery_capacity_wh) * 100.0
        remaining_battery_percent = (belief.battery_level_percent - estimated_drain_percent)

        predictions["estimated_energy_wh"] = round(estimated_energy_wh, 2)
        predictions["estimated_battery_drain_percent"] = round(estimated_drain_percent, 2)
        predictions["remaining_battery_percent"] = round(remaining_battery_percent, 2)
        
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": (estimated_drain_percent < 15.0 and remaining_battery_percent >= belief.battery_reserve_percent)
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome

    def record_trace(self, asked_utterance: str, goal: GoalSpecification, belief: BeliefState, 
                     proposal: ParameterPacket, warrant_result: Dict[str, Any], 
                     observation: Observation, committed: bool, execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        record = {
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        }
        self.audit_trail.append(record)


## Question 5.1
def audit_analyzer(audit_trail: List[Dict[str, Any]]) -> None:
    """analyzes recorded episodes and evaluates prediction accuracy"""

    rejected = 0
    approved = 0
    rejection_reasons = []
    force_errors = []

    for record in audit_trail:

        committed = record.get("committed", {})
        warrant = committed.get("warrant_verdict", {})

        if committed.get("action_approved") is False:
            rejected += 1

            reason = warrant.get("failed_criterion")
            if reason:
                rejection_reasons.append(reason)

        elif committed.get("action_approved") is True:
            approved += 1
            predicted = warrant.get("predictions", {}).get("predicted_force_n")
            outcome = record.get("execution_outcome") or {}
            actual = outcome.get("actual_force_n")

            if predicted is not None and actual is not None:
                force_errors.append(float(actual) - float(predicted))

    criterion_counts = Counter(rejection_reasons)
    most_frequent = (criterion_counts.most_common(1)[0] if criterion_counts else None)

    print("\n=== POST-HOC EPISTEMIC AUDIT ===")

    print(f"Rejected proposals: {rejected}")
    print(f"Approved executions: {approved}")

    if most_frequent:
        print(
            f"Most frequent rejection criterion: "
            f"{most_frequent[0]} ({most_frequent[1]} times)"
        )
    else:
        print("Most frequent rejection criterion: None")

    if len(force_errors) >= 2:
        force_variance = pvariance(force_errors)
        print(
            f"Variance of force prediction errors: "
            f"{force_variance:.4f} N²"
        )

    elif len(force_errors) == 1:
        print("Force prediction variance: Insufficient data (only one execution).")

    else:
        print("Force prediction variance: No actual force measurements available.")

    print(f"Force comparisons available: {len(force_errors)}")



# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    proposer = Proposer(seed=7)
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: PROPOSAL AND VALIDATION (using the Proposer) ===")

    max_attempts = 10
    
    for attempt in range(max_attempts):
        print(f"\n--- Attempt {attempt + 1} ---")
        proposal = proposer.propose(goal)
    
        print(f"Proposed velocity: {proposal.max_velocity_mps:.3f} m/s")
        print(f"Proposed tilt rate: {proposal.tilt_rate_radps:.3f} rad/s")
    
        warrant_result = warrant_gate.validate(
            proposal, goal, belief_state
        )
        print("Warrant checks:", warrant_result["warrant_checks"])
    
        if not warrant_result["passed"]:
            print("REJECTED:", warrant_result["failed_criterion"])  
            executive.record_trace(
                raw_command,
                goal,
                belief_state,
                proposal,
                warrant_result,
                raw_observation,
                committed=False
            )
            continue   
        print("APPROVED: Proceeding with execution.")
    
        outcome = executive.execute_action(
            proposal, goal, warrant_result
        )
    
        executive.record_trace(
            raw_command,
            goal,
            belief_state,
            proposal,
            warrant_result,
            raw_observation,
            committed=True,
            execution_outcome=outcome
        )
        print("Execution outcome:", outcome)
        break
    else:
        print("No proposal passed the warrant gate.")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

    ## Post-hoc analysis (Question 5.1)
    audit_analyzer(executive.audit_trail)

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793663385.3199422, measured_temperature_c=21.87, measured_friction=0.42, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.87°C, Friction=0.42

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: PROPOSAL AND VALIDATION (using the Proposer) ===

--- Attempt 1 ---
Proposed velocity: 0.332 m/s
Proposed tilt rate: 0.017 rad/s
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True, 'energetically_viable': True}
REJECTED: safe

--- Attempt 2 ---
Proposed velocity: 0.334 m/s
Proposed tilt rate: 0.014 rad/s
Warrant checks: {'grounded': True, 'feasible': True, 'safe': Fa

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.
* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.
* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

------------
### Question 1.1:

| Integration Diagram block | Python class / method | Responsibility |
|---|---|---|
| **Interpret & ground** | `InterpretationSystem.interpret()` | Converts Eva's request into a formal `GoalSpecification`. |
| **Perceive** | `SensingSystem.sense()` + `PerceptionSystem.update()` | Collects sensor observations and updates the internal `BeliefState`. |
| **Propose** | `ParameterPacket` (the `Proposer` class is currently incomplete) | Represents candidate execution parameters, such as grasp pose, velocity, and tilt rate. |
| **Know & predict** | `SemanticDigitalTwin.predict_properties()` | Predicts the physical consequences of a proposed action. |
| **Decide: warrant gate** | `WarrantGate.validate()` | Checks grounding, feasibility, safety, monitoring, repairability, and recordability before approving an action. |
| **Act & regulate** | `ExecutionEngine.execute_action()` | Simulates the execution of an approved action and evaluates its outcome. |
| **Record & justify** | `ExecutionEngine.record_trace()` | Records the request, assumptions, observations, decisions, and execution outcome. |

**Data flow:** The robot first observes its environment and updates its `BeliefState`. The `InterpretationSystem` combines this information with Eva's request to produce a `GoalSpecification`.. a candidate action is then represented by a `ParameterPacket` which then its predicted consequences are evaluated by the `SemanticDigitalTwin`. The `WarrantGate` checks whether the proposal satisfies the required criteria.. if approved, the `ExecutionEngine` executes the action and records the outcome.. if rejected, another proposal can be generated and checked.



### Question 1.2:

Code inserted above.. justification: I chose a **truncated Gaussian distribution** because the Proposer needs to generate different execution parameters while keeping them concentrated around reasonable nominal values.. like for example.. the robot should generally prefer moderate movement velocities and tilt rates rather than frequently generating extreme values.

The Gaussian distribution allows us to control this behaviour through its mean and standard deviation.. BUTTT since its theoretical range is unlimited.. I used truncated sampling to ensure that the generated parameters remain within predefined physical limits.

I also kept the velocity and tilt rate independently sampled for simplicity.. while leaving the WarrantGate responsible for evaluating their combined physical consequences.

This approach provides a balance between flexibility and controlled exploration.. allowing the robot to generate alternative proposals without executing them before they have been validated.



### Question 2.1:

Modified the `SemanticDigitalTwin.predict_properties` method as specified..


### Question 2.2:

Test Case:::

In [10]:
proposer = Proposer(
    velocity_mean=0.35,
    velocity_std=0.001,
    tilt_mean=0.015,
    tilt_std=0.0005,
    contact_depth_m=0.03,
    seed=7
)

proposal = proposer.propose(goal)
result = warrant_gate.validate(proposal, goal, belief_state)

print("Proposed velocity:", proposal.max_velocity_mps)
print("Predicted force:", result["predictions"]["predicted_force_n"])
print("Warrant checks:", result["warrant_checks"])
print("Gate verdict:", result["passed"])

assert round(proposal.max_velocity_mps, 2) == 0.35
assert result["predictions"]["predicted_force_n"] > goal.max_allowable_force_n
assert result["passed"] is False
assert result["failed_criterion"] == "safe"

print("SUCCESS: Aggressive proposal rejected before execution.")

Proposed velocity: 0.3497441197115524
Predicted force: 4.81
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate verdict: False
SUCCESS: Aggressive proposal rejected before execution.


----------------
### Question 3.1:

related code modified..


### Question 3.2:

`WarrantGate` testtt:::

In [16]:
## --- Question 3.2: Temperature and Fluid Sloshing Test ---

belief_state.ambient_temperature_c = 28.0

test_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.2,
    tilt_rate_radps=0.15,
    contact_depth_m=0.0
)

test_result = warrant_gate.validate(test_proposal, goal, belief_state)

print("\n=== Question 3.2: HOT ENVIRONMENT TEST ===")
print("Effective viscosity:", 1.2 * (1.0 - 0.01 * (28.0 - 20.0)), "Pa.s")

print("Predicted wave height:", test_result["predictions"]["predicted_wave_mm"], "mm")

print("Warrant checks:", test_result["warrant_checks"])
print("Gate verdict:", test_result["passed"])

assert test_result["predictions"]["predicted_wave_mm"] > goal.max_slosh_wave_mm
assert test_result["passed"] is False
assert test_result["failed_criterion"] == "safe"

print("SUCCESS: Warrant Gate rejected the proposal due to excessive sloshing.")


=== Question 3.2: HOT ENVIRONMENT TEST ===
Effective viscosity: 1.104 Pa.s
Predicted wave height: 6.87 mm
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate verdict: False
SUCCESS: Warrant Gate rejected the proposal due to excessive sloshing.


At 28°C.. the effective viscosity decreases from 1.2 to 1.104 Pa.s.. resulting in a higher predicted wave height of approximately 6.87 mm for a tilt rate of 0.15 rad/s.. since this exceeds the allowable limit of 1.5 mm.. the `WarrantGate` triggers a safety violation and rejects the proposal before execution.

--------------
### Question 4.1:

code above modified with an illustrative assumption of 1 Wh per metre of movement and a total battery capacity of 100 Wh...



### Question 4.2:

Battery ADD testtt:::

In [18]:
## --- Question 4.2: LOW BATTERY TEST ---

belief_state.battery_level_percent = 12.0

test_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.2,
    tilt_rate_radps=0.015,
    contact_depth_m=0.01,
    trajectory_distance_m=10.0
)

result = warrant_gate.validate(test_proposal, goal, belief_state)

print("\n=== Question 4.2: LOW BATTERY TEST ===")

print("Estimated energy consumption:", result["predictions"]["estimated_energy_wh"], "Wh")
print("Estimated battery drain:", result["predictions"]["estimated_battery_drain_percent"], "%")
print("Remaining battery:", result["predictions"]["remaining_battery_percent"], "%")

print("Warrant checks:", result["warrant_checks"])
print("Gate verdict:", result["passed"])

if not result["passed"] and result["failed_criterion"] == "energetically_viable":

    print("\n[EXECUTIVE] Coffee delivery postponed.")
    print("[EXECUTIVE] Insufficient battery reserve.")
    print("[EXECUTIVE] Requesting recharge before continuing.")

    recovery_action = "request_recharge"

    assert recovery_action == "request_recharge"
    assert result["warrant_checks"]["energetically_viable"] is False

    print("SUCCESS: System redirected to recharge.")

else:
    print("Unexpected result: Energy check did not trigger.")


=== Question 4.2: LOW BATTERY TEST ===
Estimated energy consumption: 10.0 Wh
Estimated battery drain: 10.0 %
Remaining battery: 2.0 %
Warrant checks: {'grounded': True, 'feasible': True, 'safe': True, 'monitorable': True, 'repairable': True, 'recordable': True, 'energetically_viable': False}
Gate verdict: False

[EXECUTIVE] Coffee delivery postponed.
[EXECUTIVE] Insufficient battery reserve.
[EXECUTIVE] Requesting recharge before continuing.
SUCCESS: System redirected to recharge.


----------------------
### Question 5.1:

`audit_analyzer(audit_trail)` is implemented and demonstrated in the code above and the outputs...



### Question 5.2:

Maintaining separate records for `asked`, `belief`, `committed`, and `observed` makes the robot's decision-making process more transparent and easier to evaluate.

Pure data-driven imitation learning can learn to reproduce actions based on previously observed examples, BUT it doesn't necessarily explain why a particular action was chosen, what the robot believed about the situation, or what evidence supported that decision.

By keeping these records separate, we can distinguish between what the user originally requested (`asked`), what the robot believed about the environment (`belief`), what its sensors actually observed (`observed`), and what action it decided to approve or reject and why (`committed`).

This distinction matters because the robot's beliefs might not always match the actual sensor observations. Without these separate records, it would be difficult to identify whether a failure came from incorrect perception, an outdated belief, an unsuitable decision, or unexpected physical conditions.

In other words, the audit trail allows us to understand not only WHAT the robot did, but also WHAT it was asked to do, WHAT it believed, WHAT it observed, and WHY it made that decision. This makes its behaviour more traceable and allows us to evaluate whether its decisions were justified rather than simply judging whether its final actions succeeded. 

**Anddd the records don't automatically prove that the robot's reasoning was correct. They make its assumptions, observations, and decisions inspectable, which is what allows us to evaluate and question them.**

*(I wrote this report using AI)*

----------------------------------

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](./robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---

### ROBOT ANATOMYYY

The robot consists of several functional components that work together to support perception, movement and interaction with the environment.

- **Links vs Joints:** Links are the rigid segments of the robot.. while joints connect these segments and allow relative movement.. such as at the shoulder and elbow.

- **Actuators vs Sensors:** Actuators generate the force and movement needed to operate the robot.. while sensors collect information about its body and surroundings.

- **Effectors vs Bases vs Torsos vs Heads:** Effectors interact directly with the environment.. such as the gripper. The base provides mobility and support, the torso forms the main body structure and the head can carry sensors and cameras.

- **Wheels vs Legs vs Grippers:** Wheels provide ground mobility, legs allow walking and grippers are used to grasp and manipulate objects.. the pictured robot uses wheels rather than legs.

- **Proprioception vs Exteroception vs Interoception:** Proprioception provides information about the robot's own position and movement.. exteroception senses the external environment through devices such as cameras and LiDAR and interoception monitors internal conditions such as battery level and motor temperature.

Together.. these components allow the robot to perceive its environment, understand its own physical state and execute actions through its body.. within the embodied cognitive architecture.. sensors provide information to the Manipulation Executive through perception.. while actuators, joints, wheels and grippers allow it to execute actions through the *Body–World System*.. this connects the robot's internal decision-making with its physical interaction with the environment.

------------------

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 